<a href="https://colab.research.google.com/github/silvanarayunda96/CorountinesDemo/blob/main/Fase1_RAG_Pipeline_Initial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install -q langchain langchain-text-splitters langchain-community pypdf

In [5]:
!pip install -q langchain langchain-community langchain-huggingface chromadb pypdf sentence-transformers

PDF Parsing & Cleaning

In [6]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "kurikulum_sr.pdf"
loader = PyPDFLoader(file_path)
pages = loader.load()

print(f"Berhasil membedah {len(pages)} halaman PDF.")
print(f"Contoh isi halaman 1:\n{pages[0].page_content[:500]}...")

Berhasil membedah 122 halaman PDF.
Contoh isi halaman 1:
KURIKULUM SEKOLAH RAKYAT...


Pemotongan Materi (Chunking)

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150
)

chunks = text_splitter.split_documents(pages)

print(f"Total potongan materi (chunks): {len(chunks)}")

Total potongan materi (chunks): 287


Embedding Model & vector DB

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

model_name = "LazarusNLP/all-indo-e5-small-v4"
model_kwargs = {'device': 'cuda'}
encode_kwargs = {'normalize_embeddings': True}

embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

vector_db = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./db_sekolah_rakyat"
)

print("Database RAG berhasil dibuat secara lokal!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/176 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/965 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Database RAG berhasil dibuat secara lokal!


Demo Pencarian (Retrieval)

In [9]:
pertanyaan = "Apa visi utama dari kurikulum Sekolah Rakyat?"

hasil = vector_db.similarity_search(pertanyaan, k=3)

print(f"Pertanyaan: {pertanyaan}\n")
print("Materi yang ditemukan AI:")
for i, teks in enumerate(hasil):
    print(f"--- Potongan {i+1} ---\n{teks.page_content}\n")

Pertanyaan: Apa visi utama dari kurikulum Sekolah Rakyat?

Materi yang ditemukan AI:
--- Potongan 1 ---
Ruang lingkup kurikulum ini mencakup tujuan, materi, metoda, dan evaluasi yang 
diimplementasikan di Sekolah Rakyat. Selain itu, berbagai aspek teknis seperti 
penyelenggaraan Sekolah Rakyat, mulai dari kurikulum satuan pendidikan, rekrutmen 
murid dan tenaga pendidik, tata kelola, komunikasi publik, hingga mekanisme 
pengawasan dan anggaran. Dengan adanya dokumen kurikulum ini, diharapkan seluruh 
pihak yang terlibat dalam penyelenggaraan Sekolah Rakyat  dapat memiliki acuan yang 
jelas dalam menjalankan tugas dan tanggung jawabnya. 
Kami menyampaikan apresiasi dan terima kasih kepada semua pihak yang telah 
berkontribusi dalam penyusunan kurikulum ini. Semoga dokumen ini dapat menjadi 
panduan yang bermanfaat dalam mewujudkan Sekolah Rakyat yang berkualitas, inklusif, 
dan berdaya guna bagi masyarakat. Kami juga terbuka terhadap saran dan masukan 
demi penyempurnaan program ini di 